# 08 - Reference: SQL and Java to Pandas, plus Practice Problems

**This notebook:** the lookup tables you will come back to, side-by-side SQL and Pandas that actually run, a Java Streams mapping, and 12 graded practice problems with solutions.

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)

DATA = next(p for p in [Path("../data"), Path("data"), Path("study/data")] if p.exists())
orders    = pd.read_csv(DATA / "orders.csv", parse_dates=["order_ts"])
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
items     = pd.read_csv(DATA / "order_items.csv")

# Load the same data into SQLite so we can run both sides and compare
conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False)
customers.to_sql("customers", conn, index=False)
items.to_sql("order_items", conn, index=False)

def sql(q):
    return pd.read_sql(q, conn)

print("tables loaded:", sql("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist())

## 1. SQL to Pandas - the full mapping

| SQL | Pandas |
|---|---|
| `SELECT *` | `df` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `SELECT a AS x` | `df.rename(columns={"a": "x"})` |
| `SELECT a * 2 AS d` | `df.assign(d=df["a"] * 2)` |
| `SELECT DISTINCT a` | `df["a"].drop_duplicates()` / `df.drop_duplicates(subset=["a"])` |
| `WHERE a > 5` | `df[df["a"] > 5]` |
| `WHERE a > 5 AND b = 'x'` | `df[(df["a"] > 5) & (df["b"] == "x")]` |
| `WHERE a IN (1,2)` | `df[df["a"].isin([1, 2])]` |
| `WHERE a BETWEEN 1 AND 9` | `df[df["a"].between(1, 9)]` |
| `WHERE a IS NULL` | `df[df["a"].isna()]` |
| `WHERE s LIKE '%abc%'` | `df[df["s"].str.contains("abc", na=False)]` |
| `WHERE s LIKE 'abc%'` | `df[df["s"].str.startswith("abc", na=False)]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `LIMIT 10` | `df.head(10)` |
| `ORDER BY a DESC LIMIT 10` | `df.nlargest(10, "a")` |
| `GROUP BY k` | `df.groupby("k")` |
| `COUNT(*)` | `.size()` |
| `COUNT(col)` | `.count()` |
| `COUNT(DISTINCT col)` | `.nunique()` |
| `SUM/AVG/MIN/MAX` | `.sum()/.mean()/.min()/.max()` |
| `HAVING n > 5` | `.filter(lambda g: len(g) > 5)` or filter the aggregate |
| `INNER JOIN` | `a.merge(b, on="k")` |
| `LEFT JOIN` | `a.merge(b, on="k", how="left")` |
| `FULL OUTER JOIN` | `a.merge(b, on="k", how="outer")` |
| `CROSS JOIN` | `a.merge(b, how="cross")` |
| `UNION ALL` | `pd.concat([a, b])` |
| `UNION` | `pd.concat([a, b]).drop_duplicates()` |
| `EXCEPT` (anti-join) | `merge(..., how="left", indicator=True)` then keep `left_only` |
| `CASE WHEN` | `np.where(...)` / `np.select(...)` |
| `COALESCE(a, 0)` | `df["a"].fillna(0)` |
| `NULLIF` / `CAST` | `.replace(...)` / `.astype(...)` |
| `SUM(x) OVER (PARTITION BY k)` | `df.groupby("k")["x"].transform("sum")` |
| `ROW_NUMBER() OVER (PARTITION BY k)` | `df.groupby("k").cumcount() + 1` |
| `RANK()` / `DENSE_RANK()` | `.rank(method="min")` / `.rank(method="dense")` |
| `LAG(x)` / `LEAD(x)` | `df["x"].shift(1)` / `df["x"].shift(-1)` |
| `PIVOT` | `pd.pivot_table(...)` |
| `UNPIVOT` | `df.melt(...)` |
| `WITH cte AS (...)` | assign to a variable, or a `.pipe()` chain |

## 2. The same queries, run both ways

Each cell runs the SQL and the Pandas version and checks they agree.

In [ ]:
# Q1: filter + project + order + limit
q_sql = sql("""
    SELECT order_id, channel, amount
    FROM orders
    WHERE amount > 3000 AND status <> 'CANCELLED'
    ORDER BY amount DESC
    LIMIT 5
""")

q_pd = (orders[(orders["amount"] > 3000) & (orders["status"] != "CANCELLED")]
        .nlargest(5, "amount")[["order_id", "channel", "amount"]]
        .reset_index(drop=True))

print(q_pd.to_string(index=False))
print("\nmatch:", q_sql["order_id"].tolist() == q_pd["order_id"].tolist())

In [ ]:
# Q2: GROUP BY + HAVING
q_sql = sql("""
    SELECT channel, COUNT(*) AS n, ROUND(AVG(amount), 2) AS avg_amt
    FROM orders
    GROUP BY channel
    HAVING COUNT(*) > 100
    ORDER BY n DESC
""")

agg = (orders.groupby("channel")
       .agg(n=("order_id", "size"), avg_amt=("amount", "mean"))
       .round({"avg_amt": 2}))
q_pd = agg[agg["n"] > 100].sort_values("n", ascending=False).reset_index()

print(q_pd.to_string(index=False))
print("\nmatch:", q_sql["channel"].tolist() == q_pd["channel"].tolist())

In [ ]:
# Q3: JOIN + aggregate
q_sql = sql("""
    SELECT c.tier, COUNT(DISTINCT o.order_id) AS n_orders, ROUND(SUM(o.amount), 2) AS revenue
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'DELIVERED'
    GROUP BY c.tier
    ORDER BY revenue DESC
""")

q_pd = (orders.query("status == 'DELIVERED'")
        .merge(customers, on="customer_id", how="inner", validate="m:1")
        .groupby("tier", dropna=False)
        .agg(n_orders=("order_id", "nunique"), revenue=("amount", "sum"))
        .round({"revenue": 2})
        .sort_values("revenue", ascending=False)
        .reset_index())

print("SQL:\n", q_sql.to_string(index=False))
print("\nPandas:\n", q_pd.to_string(index=False))
print("\nBoth keep the missing tier as its own group: SQL does that by default,")
print("Pandas only because of dropna=False - without it that row would disappear.")

In [ ]:
# Q4: window function
q_sql = sql("""
    SELECT order_id, channel, amount,
           ROUND(AVG(amount) OVER (PARTITION BY channel), 2) AS channel_avg,
           ROW_NUMBER() OVER (PARTITION BY channel ORDER BY amount DESC) AS rn
    FROM orders
    WHERE amount IS NOT NULL
    ORDER BY channel, rn
    LIMIT 6
""")

o = orders.dropna(subset=["amount"]).copy()
o["channel_avg"] = o.groupby("channel")["amount"].transform("mean").round(2)
o = o.sort_values(["channel", "amount"], ascending=[True, False])
o["rn"] = o.groupby("channel").cumcount() + 1
q_pd = o.head(6)[["order_id", "channel", "amount", "channel_avg", "rn"]]

print("SQL:\n", q_sql.to_string(index=False))
print("\nPandas:\n", q_pd.to_string(index=False))

In [ ]:
# Q5: three-table join
q_sql = sql("""
    SELECT i.sku, COUNT(DISTINCT o.order_id) AS orders, SUM(i.qty) AS units
    FROM order_items i
    JOIN orders o    ON o.order_id = i.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE c.tier = 'gold'
    GROUP BY i.sku
    ORDER BY units DESC, i.sku
""")

q_pd = (items
        .merge(orders[["order_id", "customer_id"]], on="order_id", validate="m:1")
        .merge(customers[["customer_id", "tier"]], on="customer_id", validate="m:1")
        .query("tier == 'gold'")
        .groupby("sku")
        .agg(orders=("order_id", "nunique"), units=("qty", "sum"))
        .sort_values(["units", "sku"], ascending=[False, True])
        .reset_index())

print(q_pd.to_string(index=False))
print("\nmatch:", q_sql["sku"].tolist() == q_pd["sku"].tolist())

## 3. Java Streams to Pandas

| Java | Pandas |
|---|---|
| `list.stream().filter(o -> o.amt > 5)` | `df[df["amt"] > 5]` |
| `.map(Order::getAmount)` | `df["amount"]` |
| `.map(o -> o.getAmt() * 2)` | `df["amt"] * 2` |
| `.sorted(comparing(Order::getAmt).reversed())` | `df.sort_values("amt", ascending=False)` |
| `.limit(10)` | `df.head(10)` |
| `.distinct()` | `df.drop_duplicates()` |
| `.count()` | `len(df)` |
| `.mapToDouble(...).sum()` | `df["amt"].sum()` |
| `.mapToDouble(...).average()` | `df["amt"].mean()` |
| `.max(comparing(Order::getAmt))` | `df.loc[df["amt"].idxmax()]` |
| `.anyMatch(p)` / `.allMatch(p)` | `(df["x"] > 0).any()` / `.all()` |
| `Collectors.groupingBy(Order::getCh)` | `df.groupby("ch")` |
| `groupingBy(f, counting())` | `df.groupby("ch").size()` |
| `groupingBy(f, summingDouble(g))` | `df.groupby("ch")["amt"].sum()` |
| `groupingBy(f, averagingDouble(g))` | `df.groupby("ch")["amt"].mean()` |
| `Collectors.partitioningBy(p)` | `df.groupby(df["x"] > 0)` |
| `Collectors.toMap(k, v)` | `df.set_index("k")["v"].to_dict()` |
| `Collectors.joining(", ")` | `", ".join(df["s"])` or `df.groupby("k")["s"].agg(", ".join)` |
| `flatMap` | `df.explode("list_col")` |
| `IntStream.range(0, n)` | `np.arange(n)` |
| `Optional<T>` | a missing value: `NaN` / `pd.NA` |
| `BigDecimal` for money | `float64`, or `int64` in minor units |

In [ ]:
# explode = flatMap
tags = pd.DataFrame({"order_id": [1, 2], "tags": [["fast", "gift"], ["bulk"]]})
print("before:\n", tags.to_string(index=False))
print("\nafter explode:\n", tags.explode("tags").to_string(index=False))

In [ ]:
# groupingBy + joining
print(items.groupby("order_id")["sku"].agg(lambda s: ", ".join(sorted(set(s)))).head(4))
print()
# partitioningBy
print(orders.groupby(orders["amount"] > 1500)["order_id"].size().rename({False: "small", True: "large"}))

## 4. NumPy to Pandas quick reference

| Task | NumPy | Pandas |
|---|---|---|
| build | `np.array([...])` | `pd.Series([...])`, `pd.DataFrame({...})` |
| shape | `arr.shape` | `df.shape` |
| type | `arr.dtype` | `df.dtypes` |
| select by position | `arr[2]`, `arr[1:3]` | `s.iloc[2]`, `df.iloc[1:3]` |
| select by label | - | `s.loc["x"]`, `df.loc["x"]` |
| filter | `arr[arr > 5]` | `df[df["a"] > 5]` |
| element-wise math | `arr * 2` | `df["a"] * 2` |
| missing | `np.nan`, `np.isnan` | `NaN`/`pd.NA`, `.isna()` |
| aggregate | `arr.sum(axis=0)` | `df.sum()` (column-wise by default) |
| group | `np.bincount`, `np.add.at` | `df.groupby(...)` |
| sort | `np.sort`, `np.argsort` | `df.sort_values` |
| unique | `np.unique` | `s.unique()`, `s.value_counts()` |
| join | `np.intersect1d` | `df.merge` |
| convert | - | `df.to_numpy()`, `pd.DataFrame(arr)` |

Rule of thumb: **NumPy for homogeneous numeric matrices and maths; Pandas for heterogeneous labelled tables.** Drop to `.to_numpy()` inside a hot numeric loop, then come back.

## 5. Practice problems

Work through these in order. Solutions follow in the next cell - resist looking.

**Warm-up**
1. How many orders are there per status, as a percentage of all orders, rounded to 1 decimal?
2. List the 3 cities with the highest average order amount (delivered orders only), showing the average and the order count.
3. Add a column `weekday` and find which weekday has the most cancelled orders.

**Intermediate**
4. For each customer, compute total spend, order count, and average order value. Return the top 5 by total spend, with the customer's name and city.
5. Find customers whose **second** order was larger than their first.
6. What share of each channel's revenue comes from the top 20% of its orders by amount?
7. Compute a month-over-month revenue growth table per channel, wide format (rows = month, columns = channel, values = growth percent).

**Advanced**
8. For each order, add the number of days since that customer's previous order (`NaN` for their first).
9. Identify customers whose spending in the second half of the data is more than 50% below their first half (churn risk).
10. Build a cohort table: rows = signup month, columns = months since signup, values = number of active customers who ordered in that month.
11. For each SKU, find the customer tier that buys the most units of it.
12. Using a 7-day rolling window per channel, find the date on which each channel hit its highest 7-day revenue.

In [ ]:
# ---------- Warm-up solutions ----------
print("1.")
print((orders["status"].value_counts(normalize=True) * 100).round(1))
print()

print("2.")
print(orders.query("status == 'DELIVERED'")
      .merge(customers, on="customer_id", validate="m:1")
      .groupby("city")
      .agg(avg_amount=("amount", "mean"), n=("order_id", "count"))
      .round({"avg_amount": 2})
      .nlargest(3, "avg_amount")
      .to_string())
print()

print("3.")
print(orders.assign(weekday=lambda d: d["order_ts"].dt.day_name())
      .query("status == 'CANCELLED'")["weekday"]
      .value_counts().head(3).to_string())

In [ ]:
# ---------- Intermediate solutions ----------
print("4.")
cust_stats = (orders.groupby("customer_id")
              .agg(total_spend=("amount", "sum"),
                   n_orders=("order_id", "count"),
                   avg_order=("amount", "mean"))
              .round(2)
              .merge(customers[["customer_id", "name", "city"]], on="customer_id")
              .nlargest(5, "total_spend"))
print(cust_stats[["name", "city", "n_orders", "total_spend", "avg_order"]].to_string(index=False))
print()

print("5.")
seq = orders.sort_values("order_ts").copy()
seq["order_seq"] = seq.groupby("customer_id").cumcount() + 1
first_two = seq[seq["order_seq"] <= 2].pivot(index="customer_id", columns="order_seq", values="amount")
grew = first_two[first_two[2] > first_two[1]]
print("customers whose 2nd order beat their 1st:", len(grew), "of", len(first_two.dropna()))
print(grew.head(4).round(2).to_string())
print()

print("6.")
def top20_share(g):
    g = g.dropna(subset=["amount"])
    cutoff = g["amount"].quantile(0.80)
    return g.loc[g["amount"] >= cutoff, "amount"].sum() / g["amount"].sum() * 100
print(orders.groupby("channel")[["amount"]].apply(top20_share).round(1).rename("top20_share_pct").to_string())
print()

print("7.")
mrev = (orders.assign(month=lambda d: d["order_ts"].dt.to_period("M").astype(str))
        .groupby(["month", "channel"])["amount"].sum().unstack(fill_value=0))
growth = (mrev.pct_change() * 100).round(1)
print(growth.head(5).to_string())

In [ ]:
# ---------- Advanced solutions ----------
print("8.")
seq = orders.sort_values(["customer_id", "order_ts"]).copy()
seq["prev_order_ts"] = seq.groupby("customer_id")["order_ts"].shift(1)
seq["days_since_prev"] = (seq["order_ts"] - seq["prev_order_ts"]).dt.days
print(seq[["customer_id", "order_ts", "days_since_prev"]].head(6).to_string(index=False))
print("median gap between orders:", seq["days_since_prev"].median(), "days")
print()

print("9.")
midpoint = orders["order_ts"].min() + (orders["order_ts"].max() - orders["order_ts"].min()) / 2
half = orders.assign(period=lambda d: np.where(d["order_ts"] < midpoint, "h1", "h2"))
pv = half.pivot_table(index="customer_id", columns="period", values="amount", aggfunc="sum", fill_value=0)
pv["change_pct"] = ((pv["h2"] - pv["h1"]) / pv["h1"].replace(0, np.nan) * 100).round(1)
churn = pv[pv["change_pct"] < -50].sort_values("change_pct")
print("churn-risk customers:", len(churn))
print(churn.head(5).round(0).to_string())
print()

print("10.")
cohort_base = (orders.merge(customers[["customer_id", "signup_date"]], on="customer_id", validate="m:1")
               .assign(cohort=lambda d: d["signup_date"].dt.to_period("M"),
                       order_month=lambda d: d["order_ts"].dt.to_period("M")))
cohort_base["months_since"] = ((cohort_base["order_month"] - cohort_base["cohort"])
                               .apply(lambda x: x.n))
# keep only orders placed on or after signup, and only the first 8 months of each cohort
cohort_base = cohort_base[cohort_base["months_since"].between(0, 7)]
cohort = cohort_base.pivot_table(index="cohort", columns="months_since",
                                 values="customer_id", aggfunc="nunique", fill_value=0)
print(cohort.tail(6).to_string())
print("(read a row as: of the customers who signed up in this month, how many ordered")
print(" 0, 1, 2 ... months later. Real cohort tables usually show this as a retention %.)")
print()

print("11.")
sku_tier = (items
            .merge(orders[["order_id", "customer_id"]], on="order_id", validate="m:1")
            .merge(customers[["customer_id", "tier"]], on="customer_id", validate="m:1")
            .groupby(["sku", "tier"], dropna=False)["qty"].sum().reset_index())
best = sku_tier.loc[sku_tier.groupby("sku")["qty"].idxmax()]
print(best.to_string(index=False))
print("(bronze wins everywhere here simply because it is the biggest tier -")
print(" for a fair comparison, divide by the number of customers in each tier.)")
per_cust = sku_tier.merge(customers.groupby("tier", dropna=False).size().rename("n_cust").reset_index(),
                          on="tier", how="left")
per_cust["qty_per_customer"] = (per_cust["qty"] / per_cust["n_cust"]).round(2)
fair = per_cust.loc[per_cust.groupby("sku")["qty_per_customer"].idxmax()]
print()
print(fair[["sku", "tier", "qty", "n_cust", "qty_per_customer"]].to_string(index=False))
print()

print("12.")
daily_rev = (orders.set_index("order_ts").groupby("channel")
             .resample("D")["amount"].sum().rename("revenue").reset_index())
daily_rev["roll7"] = (daily_rev.groupby("channel")["revenue"]
                      .transform(lambda s: s.rolling(7, min_periods=7).sum()))
peak = daily_rev.loc[daily_rev.groupby("channel")["roll7"].idxmax()]
print(peak.round({"revenue": 0, "roll7": 0}).to_string(index=False))

In [ ]:
conn.close()
print("done")

## 6. Where to go next

**Practise on real data.** Any of these will exercise everything in this track:
- your own Postgres: `pd.read_sql(...)` then aggregate in Pandas and compare against the SQL answer
- public datasets: NYC taxi trips (big, time series), any Kaggle CSV

**Adjacent tools worth an afternoon each**
- **Polars** - the same ideas with a stricter API, multi-threaded and lazy. If you like Java Streams you will like its expression API.
- **DuckDB** - run SQL directly over Parquet and CSV files, and hand the result to Pandas with `.df()`. Often the shortest path when you already think in SQL.
- **Matplotlib / Seaborn / Plotly** - `df.plot()` covers most quick charts.
- **scikit-learn** - it takes NumPy arrays and DataFrames directly, so notebooks 01-07 are the prerequisite.

**Habits that pay off**
1. Check `df.info()` and `df.isna().sum()` on every new dataset before anything else.
2. Pass `validate=` on every merge.
3. Name your aggregates with `agg(name=("col", "fn"))`.
4. Prefer method chains over a trail of intermediate variables; use `.pipe()` for custom steps.
5. When something is slow, look for the Python loop before looking for a bigger machine.